In [ ]:
import pandas as pd
from preprocessing_utils import (
    INTERIM_DIR, RAW_DIR, latest_version, normalize_label, normalize_well_id,
    parse_date, parse_number, save_table,
)

df = pd.read_csv(RAW_DIR / "wells.csv")
df["well_id"] = normalize_well_id(df["well_id"])
df["commissioned"] = parse_date(df["commissioned"])
for column in ["role", "zone"]:
    df[column] = normalize_label(df[column])

for column in ["x", "y", "z"]:
    df[column] = parse_number(df[column]).replace(-999, pd.NA)

df["z"] = -df["z"].abs()

In [ ]:
surveys = pd.read_parquet(INTERIM_DIR / "well_surveys.parquet")
geometry = ["x", "y", "screen_top", "screen_bottom"]
usable = surveys.loc[
    surveys["status"].eq("accepted") & surveys["survey_date"].notna()
].dropna(subset=geometry)
latest = latest_version(usable, "well_id", "survey_date").set_index("well_id")
latest["z"] = (latest["screen_top"] + latest["screen_bottom"]) / 2

for column in ["x", "y", "z"]:
    df[column] = df[column].fillna(df["well_id"].map(latest[column]))

In [ ]:
zones = pd.read_parquet(INTERIM_DIR / "zones.parquet")
for zone in zones.itertuples(index=False):
    inside = (
        df["x"].between(zone.x_min, zone.x_max, inclusive="left")
        & df["y"].between(zone.y_min, zone.y_max, inclusive="left")
    )
    df.loc[inside & df["zone"].isna(), "zone"] = zone.zone

df = df.drop_duplicates()
if not df["well_id"].is_unique:
    raise ValueError("Conflicting records for the same well_id")
if df[["well_id", "x", "y", "z"]].isna().any().any():
    raise ValueError("Missing well IDs or coordinates after survey recovery")

In [ ]:
display(df.head())
save_table(df, "wells")